In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import os
import sys

import h5py
import healpy as hp
import numpy as np

sys.path.append(os.path.join(os.getcwd(), ".."))
from scripts.utils import remove_mono_dipole, setup_logging, plot_predictions
from scripts import core, Core

from notebooks.ksw_joblib import KSW_joblib

# Monkey patch the core module
core.KSW = KSW_joblib

In [ ]:
logger = setup_logging(__name__, level=logging.DEBUG)

## Heidelberg

In [ ]:
core = Core(
    [
        "settings/heidelberg.json",
        "--nsims",
        "100",
        "--narray",
        "10",
        # "--polarizations",
        # "TE",
    ]
)

In [ ]:
from scripts.estimator import run_ksw_step

print("num cpus available:", len(os.sched_getaffinity(0)))
thetas = int(np.floor(1.5 * core.lmax + 1)) // len(os.sched_getaffinity(0))

run_ksw_step(core, thetas)

In [ ]:
fisher = float(core.ksw.compute_fisher())
fisher, np.sqrt(1 / fisher)

In [ ]:
num_estimates = 100

fnls = np.random.uniform(core.fnl_min, core.fnl_max, num_estimates)
hei_idxs = range(1, num_estimates)
hei_fnls = fnls[hei_idxs]


def alm_hei_loader(idx):
    str_idx = str(idx).zfill(4)
    base1 = f"data/heidelberg/alm_l_{str_idx}_v3.fits"
    base2 = f"data/heidelberg/alm_nl_{str_idx}_v3.fits"

    alm_heidelberg_l = np.array(hp.read_alm(base1, hdu=1))
    alm_heidelberg_nl = np.array(hp.read_alm(base2, hdu=1))
    fnl = fnls[idx]

    alm = alm_heidelberg_l + fnl * alm_heidelberg_nl
    alm *= 2.7255 * 10 ** (6)  # convert heidelberg to uK
    alm = remove_mono_dipole(alm)

    logger.info("sending idx: %s, fnl: %s" % (idx, fnl))
    return alm


hei_estimates = core.ksw.compute_estimate_batch(
    alm_hei_loader, hei_idxs, fisher=fisher, theta_batch=thetas
)

In [ ]:
plot_predictions(hei_fnls, hei_estimates, fisher=fisher)

## Sim

In [ ]:
alm_file = h5py.File(core.alm_file, "r", swmr=True, locking=False)
alms = alm_file["alm"]
fnls = alm_file["fnl"]


def estimator_loader(idx):
    """Loads in a single alm given an idx."""
    print("sending idx: %s, fnl: %s" % (idx, fnls[idx]))
    return alms[int(idx)]


hei_idxs = range(core.total_sims)
sim_estimates = core.ksw.compute_estimate_batch(
    estimator_loader, hei_idxs, fisher=fisher, theta_batch=thetas
)

In [ ]:
plot_predictions(fnls[hei_idxs], sim_estimates, fisher=fisher)

In [ ]:
from matplotlib import pyplot as plt

plt.scatter(hei_fnls, hei_estimates, label="heidelberg")
plt.scatter(fnls, sim_estimates, label="sim")

line = [min(fnls), max(fnls)]
plt.plot(line, line, color="red", linestyle="--", label="truth")
plt.xlabel("fnls")
plt.ylabel("Estimates")
plt.title("Scatter plot of Estimates vs fnls")
plt.show()